# Formative 2 — Multimodal Data Preprocessing
## Image & Audio Verification Pipeline
### Approach B: Pure Pretrained Embedding + Cosine Similarity

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image, ImageOps
import librosa
import librosa.display
import torchvision.transforms as T
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'data'   # adjust if needed
print('Setup complete ✓')

## 1. Image EDA — Load & Display Sample Faces

In [ ]:
images_dir = os.path.join(DATA_DIR, 'images')
persons = sorted(os.listdir(images_dir))

fig, axes = plt.subplots(len(persons), 3, figsize=(10, 4*len(persons)))
if len(persons) == 1:
    axes = [axes]

for row, person in enumerate(persons):
    person_dir = os.path.join(images_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.jpg','.jpeg','.png')))
    for col, fname in enumerate(files[:3]):
        img = Image.open(os.path.join(person_dir, fname)).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].set_title(f'{person} / {fname}', fontsize=9)
        axes[row][col].axis('off')

plt.suptitle('Sample Facial Images per Person', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Image Augmentation (≥2 per image)

In [ ]:
augmentations = [
    ('Original',           T.Compose([])),
    ('Rotation +20°',      T.RandomRotation((20, 20))),
    ('Horizontal Flip',    T.RandomHorizontalFlip(p=1.0)),
    ('Grayscale',          T.Grayscale(num_output_channels=3)),
    ('Color Jitter',       T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4)),
    ('Random Crop',        T.Compose([T.Resize(256), T.CenterCrop(224)])),
]

# Show augmentations on first image of each person
for person in persons:
    person_dir = os.path.join(images_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.jpg','.jpeg','.png')))
    if not files:
        continue
    img = Image.open(os.path.join(person_dir, files[0])).convert('RGB')

    fig, axes = plt.subplots(1, len(augmentations), figsize=(18, 3))
    for ax, (name, aug) in zip(axes, augmentations):
        augmented = aug(img)
        ax.imshow(augmented)
        ax.set_title(name, fontsize=8)
        ax.axis('off')
    plt.suptitle(f'Augmentations — {person}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Extract & Save Image Features → image_features.csv

In [ ]:
from feature_utils import build_image_features_csv
build_image_features_csv(DATA_DIR, out_csv='image_features.csv')

df_img = pd.read_csv('image_features.csv')
print(f'Shape: {df_img.shape}')
df_img[['person','file']].head()

## 4. Audio EDA — Waveforms & Spectrograms

In [ ]:
audio_dir = os.path.join(DATA_DIR, 'audio')
persons_audio = sorted(os.listdir(audio_dir))

for person in persons_audio:
    person_dir = os.path.join(audio_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.m4a','.wav','.mp3','.ogg','.flac')))
    for fname in files[:2]:
        y, sr = librosa.load(os.path.join(person_dir, fname), sr=22050)

        fig, axes = plt.subplots(1, 2, figsize=(12, 3))

        # Waveform
        librosa.display.waveshow(y, sr=sr, ax=axes[0])
        axes[0].set_title(f'Waveform — {person}/{fname}')
        axes[0].set_xlabel('Time (s)')
        axes[0].set_ylabel('Amplitude')

        # Mel Spectrogram
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        S_dB = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', ax=axes[1])
        axes[1].set_title(f'Mel Spectrogram — {person}/{fname}')
        fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

        plt.tight_layout()
        plt.show()

## 5. Audio Augmentation (≥2 per sample)

In [ ]:
for person in persons_audio:
    person_dir = os.path.join(audio_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.m4a','.wav','.mp3','.ogg','.flac')))
    if not files:
        continue
    y, sr = librosa.load(os.path.join(person_dir, files[0]), sr=22050)

    # Augmentation 1: Pitch shift (+3 semitones)
    y_pitch = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=3)

    # Augmentation 2: Time stretch (0.85×)
    y_time  = librosa.effects.time_stretch(y=y, rate=0.85)

    # Augmentation 3: Add Gaussian noise
    noise   = np.random.normal(0, 0.005, len(y))
    y_noise = y + noise

    fig, axes = plt.subplots(1, 4, figsize=(18, 3))
    for ax, (label, sig) in zip(axes, [
        ('Original',        y),
        ('Pitch +3st',      y_pitch),
        ('Time ×0.85',      y_time),
        ('+ Gaussian noise',y_noise),
    ]):
        librosa.display.waveshow(sig, sr=sr, ax=ax)
        ax.set_title(label, fontsize=9)
    plt.suptitle(f'Audio Augmentations — {person}/{files[0]}', fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Extract & Save Audio Features → audio_features.csv

In [ ]:
from feature_utils import build_audio_features_csv
build_audio_features_csv(DATA_DIR, out_csv='audio_features.csv')

df_aud = pd.read_csv('audio_features.csv')
print(f'Shape: {df_aud.shape}')
df_aud[['person','file']].head()

## 7. Face Verification — Evaluation

In [ ]:
from face_verifier import FaceVerifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

fv = FaceVerifier(csv_path='image_features.csv', threshold=0.55)

y_true, y_pred = [], []
for person in persons:
    person_dir = os.path.join(images_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.jpg','.jpeg','.png')))
    for fname in files:
        pred_name, score = fv.identify_from_path(os.path.join(person_dir, fname))
        y_true.append(person)
        y_pred.append(pred_name)

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
print(f'Face Verification  →  Accuracy: {acc:.3f}   F1 (weighted): {f1:.3f}')

labels = sorted(set(y_true + y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False)
plt.title('Face Verification — Confusion Matrix')
plt.tight_layout()
plt.show()

## 8. Voice Verification — Evaluation

In [ ]:
from voice_verifier import VoiceVerifier

vv = VoiceVerifier(csv_path='audio_features.csv', threshold=0.80)

y_true_v, y_pred_v = [], []
for person in persons_audio:
    person_dir = os.path.join(audio_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.m4a','.wav','.mp3','.ogg','.flac')))
    for fname in files:
        pred_name, score = vv.identify_from_path(os.path.join(person_dir, fname))
        y_true_v.append(person)
        y_pred_v.append(pred_name)

acc_v = accuracy_score(y_true_v, y_pred_v)
f1_v  = f1_score(y_true_v, y_pred_v, average='weighted', zero_division=0)
print(f'Voice Verification  →  Accuracy: {acc_v:.3f}   F1 (weighted): {f1_v:.3f}')

labels_v = sorted(set(y_true_v + y_pred_v))
cm_v = confusion_matrix(y_true_v, y_pred_v, labels=labels_v)
disp_v = ConfusionMatrixDisplay(cm_v, display_labels=labels_v)
fig, ax = plt.subplots(figsize=(6, 5))
disp_v.plot(ax=ax, colorbar=False)
plt.title('Voice Verification — Confusion Matrix')
plt.tight_layout()
plt.show()

## 9. Similarity Score Distribution — Both Modalities

In [ ]:
# Collect all face similarity scores
genuine_face, impostor_face = [], []
for person in persons:
    person_dir = os.path.join(images_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.jpg','.jpeg','.png')))
    for fname in files:
        pred_name, score = fv.identify_from_path(os.path.join(person_dir, fname))
        if pred_name == person:
            genuine_face.append(score)
        else:
            impostor_face.append(score)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(genuine_face,  bins=15, color='green', alpha=0.7, label='Genuine')
axes[0].hist(impostor_face, bins=15, color='red',   alpha=0.7, label='Impostor')
axes[0].axvline(fv.threshold, color='black', linestyle='--', label=f'Threshold={fv.threshold}')
axes[0].set_title('Face Verification — Similarity Distribution')
axes[0].set_xlabel('Cosine Similarity')
axes[0].legend()

# Same for audio
genuine_audio, impostor_audio = [], []
for person in persons_audio:
    person_dir = os.path.join(audio_dir, person)
    files = sorted(f for f in os.listdir(person_dir)
                   if f.lower().endswith(('.m4a','.wav','.mp3','.ogg','.flac')))
    for fname in files:
        pred_name, score = vv.identify_from_path(os.path.join(person_dir, fname))
        if pred_name == person:
            genuine_audio.append(score)
        else:
            impostor_audio.append(score)

axes[1].hist(genuine_audio,  bins=15, color='green', alpha=0.7, label='Genuine')
axes[1].hist(impostor_audio, bins=15, color='red',   alpha=0.7, label='Impostor')
axes[1].axvline(vv.threshold, color='black', linestyle='--', label=f'Threshold={vv.threshold}')
axes[1].set_title('Voice Verification — Similarity Distribution')
axes[1].set_xlabel('Cosine Similarity')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Product Recommendation Model
> **Placeholder** — will be implemented after tabular data merge (Task 1).

In [ ]:
# TODO: load customer_social_profiles.csv + customer_transactions.csv
# TODO: merge, feature-engineer, train XGBoost / RandomForest
# TODO: plug into auth_gate.py run_recommendation_step()
print('Product Recommendation Model — coming soon')